# 04 — Main Regressions

Clustered pooled OLS with industry and quarter fixed effects for:
- **H1**: SR ~ PreSkew + controls  
- **H2**: UR ~ LN_VIX_pre + controls  
- **H2 segmented**: decompose UR into three post-announcement windows
  (day +1, days +2/+3, days +4/+5).

Standard errors clustered by `secid`.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd

from src.config import (
    DATA_PATH, OUT_DIR, PRE_DAY, POST_DAY, MAIN_CLUSTER_VAR,
    BASE_CONTROLS, RICH_OPTION_CONTROLS, SHOCK_PROXY,
)
from src.data_utils import load_panel, build_event_level_dataset, build_h2_segment_dataset
from src.stats import run_clustered_pooled_ols, available_controls

TABLES = OUT_DIR / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)

df = load_panel(DATA_PATH)
event, _ = build_event_level_dataset(df, pre_day=PRE_DAY, post_day=POST_DAY)

base   = available_controls(event, BASE_CONTROLS)
rich   = available_controls(event, RICH_OPTION_CONTROLS)
shock  = available_controls(event, SHOCK_PROXY)
print('Event obs:', len(event))
print('Base controls available :', base)
print('Rich controls available :', rich)

Event obs: 11468
Base controls available : ['PreIV', 'LN_MRKCAP_pre', 'ret_pre', 'LN_VOL_pre', 'LN_PRC_pre']
Rich controls available : ['Dispersion_pre', 'LN_PC_OI_pre', 'LN_PC_VLM_pre', 'LN_TOTALVAR_pre', 'IMPKURT_pre', 'LN_EXPTIME_pre']


## 1. H1 and H2 pooled OLS specifications

In [2]:
pool_specs = {
    # H1: skew revision
    'H1_pool_main':   ('SR', ['PreSkew', 'LN_VIX_pre'] + base),
    'H1_pool_rich':   ('SR', ['PreSkew', 'LN_VIX_pre'] + base + rich),
    'H1_pool_absret': ('SR', ['PreSkew', 'LN_VIX_pre'] + base + rich + shock),
    # H2: uncertainty revision
    'H2_pool_main':   ('UR', ['LN_VIX_pre', 'PreSkew'] + [c for c in base if c != 'PreIV'] + ['PreIV']),
    'H2_pool_rich':   ('UR', ['LN_VIX_pre', 'PreSkew'] + [c for c in base if c != 'PreIV'] + ['PreIV'] + rich),
    'H2_pool_absret': ('UR', ['LN_VIX_pre', 'PreSkew'] + [c for c in base if c != 'PreIV'] + ['PreIV'] + rich + shock),
    # H2 robustness outcomes
    'H2_pool_norm':   ('UR_norm_by_PreIV', ['LN_VIX_pre', 'PreSkew'] + [c for c in base if c != 'PreIV'] + ['PreIV'] + rich),
    'H2_pool_IVpost': ('IV_post',          ['LN_VIX_pre', 'PreSkew'] + [c for c in base if c != 'PreIV'] + ['PreIV'] + rich),
}

coef_rows, meta_rows = [], []
for spec_name, (depvar, regressors) in pool_specs.items():
    regs = list(dict.fromkeys([r for r in regressors if r in event.columns]))
    coef, meta, _ = run_clustered_pooled_ols(
        event, depvar, regs, cluster_var=MAIN_CLUSTER_VAR, spec_name=spec_name
    )
    coef_rows.append(coef)
    meta_rows.append(meta)

pooled_results = pd.concat(coef_rows, ignore_index=True)
pooled_meta    = pd.concat(meta_rows, ignore_index=True)

pooled_results.to_csv(TABLES / 'pooled_ols_main_results.csv', index=False)
pooled_meta.to_csv(TABLES   / 'pooled_ols_main_meta.csv',    index=False)
print('Saved: pooled_ols_main_results.csv, pooled_ols_main_meta.csv')
pooled_results

Saved: pooled_ols_main_results.csv, pooled_ols_main_meta.csv


,spec,depvar,term,coef,std_err,t_stat,p_value
0,H1_pool_main,SR,PreSkew,0.631727,0.011767,53.687514,0.000000e+00
1,H1_pool_main,SR,LN_VIX_pre,-0.060910,0.004407,-13.821777,1.883608e-43
2,H1_pool_main,SR,PreIV,0.033906,0.003148,10.771649,4.685334e-27
3,H1_pool_main,SR,LN_MRKCAP_pre,-0.009657,0.001604,-6.020037,1.743768e-09
4,H1_pool_main,SR,ret_pre,-0.020028,0.029486,-0.679246,4.969823e-01
...,...,...,...,...,...,...,...
89,H2_pool_IVpost,IV_post,LN_PC_OI_pre,-0.000663,0.000676,-0.980282,3.269468e-01
90,H2_pool_IVpost,IV_post,LN_PC_VLM_pre,0.001140,0.001998,0.570353,5.684383e-01
91,H2_pool_IVpost,IV_post,LN_TOTALVAR_pre,0.133616,0.004764,28.045971,4.473043e-173
92,H2_pool_IVpost,IV_post,IMPKURT_pre,0.667097,0.043046,15.497459,3.608842e-54


## 2. Key coefficients summary (main paper table)

In [3]:
key = pooled_results[
    (pooled_results['spec'].str.startswith('H1') & (pooled_results['term'] == 'PreSkew')) |
    (pooled_results['spec'].str.startswith('H2') & (pooled_results['term'] == 'LN_VIX_pre'))
].copy()
key.to_csv(TABLES / 'main_key_results_pooled.csv', index=False)
key

,spec,depvar,term,coef,std_err,t_stat,p_value
0,H1_pool_main,SR,PreSkew,0.631727,0.011767,53.687514,0.000000e+00
7,H1_pool_rich,SR,PreSkew,0.631173,0.011843,53.293344,0.000000e+00
20,H1_pool_absret,SR,PreSkew,0.631330,0.011859,53.234371,0.000000e+00
34,H2_pool_main,UR,LN_VIX_pre,-0.155255,0.015045,-10.319199,5.770268e-25
41,H2_pool_rich,UR,LN_VIX_pre,-0.173485,0.012139,-14.291650,2.466931e-46
54,H2_pool_absret,UR,LN_VIX_pre,-0.169947,0.011543,-14.723435,4.558990e-49
68,H2_pool_norm,UR_norm_by_PreIV,LN_VIX_pre,-0.332654,0.032909,-10.108404,5.070344e-24
81,H2_pool_IVpost,IV_post,LN_VIX_pre,0.174269,0.012140,14.355041,9.906623e-47


## 3. H2 segmented mechanism analysis

In [4]:
seg_event = build_h2_segment_dataset(df)
seg_base  = available_controls(seg_event, BASE_CONTROLS)
seg_rich  = available_controls(seg_event, RICH_OPTION_CONTROLS)
seg_shock = available_controls(seg_event, SHOCK_PROXY)

seg_specs = {
    'H2SEG_UR_1':  ('UR_1',  ['LN_VIX_pre', 'PreSkew', 'PreIV'] + seg_base + seg_rich + seg_shock),
    'H2SEG_UR_23': ('UR_23', ['LN_VIX_pre', 'PreSkew', 'PreIV'] + seg_base + seg_rich + seg_shock),
    'H2SEG_UR_45': ('UR_45', ['LN_VIX_pre', 'PreSkew', 'PreIV'] + seg_base + seg_rich + seg_shock),
}

seg_rows, seg_meta_rows = [], []
for spec_name, (depvar, regressors) in seg_specs.items():
    regs = list(dict.fromkeys([r for r in regressors if r in seg_event.columns]))
    coef, meta, _ = run_clustered_pooled_ols(
        seg_event, depvar, regs, cluster_var=MAIN_CLUSTER_VAR, spec_name=spec_name
    )
    seg_rows.append(coef)
    seg_meta_rows.append(meta)

seg_results = pd.concat(seg_rows, ignore_index=True)
seg_meta    = pd.concat(seg_meta_rows, ignore_index=True)

seg_results.to_csv(TABLES / 'h2_segmented_results.csv', index=False)
seg_meta.to_csv(TABLES    / 'h2_segmented_meta.csv',    index=False)
print('Saved: h2_segmented_results.csv, h2_segmented_meta.csv')
seg_results[seg_results['term'] == 'LN_VIX_pre']

Saved: h2_segmented_results.csv, h2_segmented_meta.csv


,spec,depvar,term,coef,std_err,t_stat,p_value
0,H2SEG_UR_1,UR_1,LN_VIX_pre,-0.169464,0.011582,-14.632030,1.754720e-48
14,H2SEG_UR_23,UR_23,LN_VIX_pre,0.121981,0.009394,12.985414,1.480357e-38
28,H2SEG_UR_45,UR_45,LN_VIX_pre,0.075929,0.009563,7.939680,2.027040e-15
